# 002 — Architecture of Sandbox Backend

学习目标：

1. 理解 Protocol + ABC 的接口设计
2. 实现 `ExecutionResult`、`SandboxBackend` Protocol
3. 实现简化版 `SandboxConfig` 和 `SandboxType`
4. 实现注册表模式和工厂函数
5. 对比 Clawith 真实实现

---


## 1. 先理解"接口"的价值

沙盒系统支持多种后端（subprocess、Docker、E2B……）。  
如果不抽象接口，`agent_tools.py` 就要写满 `if sandbox_type == "docker": ... elif sandbox_type == "e2b"`。

Clawith 的做法：定义 `SandboxBackend` Protocol，**所有后端实现同一个接口**。

> Python 的 Protocol 是"鸭子类型"——只要一个类有 `execute`、`health_check`、`get_capabilities` 方法，它就是 `SandboxBackend`。


In [2]:
from dataclasses import dataclass
from typing import Protocol, runtime_checkable
from abc import ABC, abstractmethod


@dataclass
class ExecutionResult:
    """代码执行结果。"""
    success: bool
    stdout: str
    stderr: str
    exit_code: int
    duration_ms: int
    error: str | None = None

    def __repr__(self):
        return (
            f"ExecutionResult(success={self.success}, "
            f"exit_code={self.exit_code}, "
            f"stdout_len={len(self.stdout)}, "
            f"stderr_len={len(self.stderr)}, "
            f"duration={self.duration_ms}ms)"
        )


@runtime_checkable
class SandboxBackend(Protocol):
    """所有沙盒后端必须实现的接口。"""
    name: str

    async def execute(
        self, code: str, language: str,
        timeout: int = 30, work_dir: str | None = None,
        **kwargs
    ) -> ExecutionResult:
        ...

    async def health_check(self) -> bool:
        ...

    def get_capabilities(self) -> dict:
        ...


print("✅ Protocol + ExecutionResult 定义完成")
result = ExecutionResult(True, "", "", 0, 0)
print(f"ExecutionResult 实例占用内存约 {result.__sizeof__()} bytes")


✅ Protocol + ExecutionResult 定义完成
ExecutionResult 实例占用内存约 16 bytes


## 2. 配置模型：SandboxType + SandboxConfig

沙盒的行为由配置控制。Clawith 用 `SandboxType` 枚举后端类型，用 `SandboxConfig` 集中管理所有参数：


In [3]:
from enum import Enum

class SandboxType(str, Enum):
    SUBPROCESS = "subprocess"
    DOCKER = "docker"
    E2B = "e2b"

    def __repr__(self):
        return f"SandboxType.{self.name}"


class SandboxConfig:
    """沙盒配置：控制后端行为和资源限制。"""
    def __init__(
        self,
        sandbox_type: SandboxType = SandboxType.SUBPROCESS,
        cpu_limit: str = "0.5",
        memory_limit: str = "256m",
        allow_network: bool = False,
        default_timeout: int = 30,
        max_timeout: int = 60,
        api_key: str = "",
    ):
        self.sandbox_type = sandbox_type
        self.cpu_limit = cpu_limit
        self.memory_limit = memory_limit
        self.allow_network = allow_network
        self.default_timeout = default_timeout
        self.max_timeout = max_timeout
        self.api_key = api_key

    def __repr__(self):
        return (
            f"SandboxConfig(type={self.sandbox_type.value}, "
            f"network={'on' if self.allow_network else 'off'}, "
            f"cpu={self.cpu_limit}, mem={self.memory_limit})"
        )


# 默认配置
default_config = SandboxConfig()
print(f"默认配置: {default_config}")

# 允许网络的配置
network_config = SandboxConfig(allow_network=True)
print(f"网络配置: {network_config}")


默认配置: SandboxConfig(type=subprocess, network=off, cpu=0.5, mem=256m)
网络配置: SandboxConfig(type=subprocess, network=on, cpu=0.5, mem=256m)


## 3. 注册表 + 工厂

当你新增一种沙盒后端时，只需要做两件事：

1. 写一个类实现 `SandboxBackend` Protocol
2. 注册到注册表

调用方不需要知道具体类的名字——它只知道 `SandboxType` 枚举。


In [4]:
# 注册表
_BACKEND_REGISTRY: dict[SandboxType, type] = {}

def register_backend(sandbox_type: SandboxType, backend_class: type):
    _BACKEND_REGISTRY[sandbox_type] = backend_class

def get_sandbox_backend(config: SandboxConfig):
    """工厂函数：根据配置创建对应的后端实例。"""
    backend_class = _BACKEND_REGISTRY.get(config.sandbox_type)
    if backend_class is None:
        raise ValueError(f"Unknown sandbox type: {config.sandbox_type}")
    return backend_class(config)


# 先注册一个占位后端，验证注册表工作
class PlaceholderBackend:
    name = "placeholder"
    def __init__(self, config: SandboxConfig):
        self.config = config
    async def execute(self, code, language, **kw):
        return ExecutionResult(True, f"[{self.name}] would execute {language} code", "", 0, 0)
    async def health_check(self):
        return True
    def get_capabilities(self):
        return {"languages": [self.name]}

register_backend(SandboxType.SUBPROCESS, PlaceholderBackend)

config = SandboxConfig()
backend = get_sandbox_backend(config)
print(f"工厂返回: {backend.__class__.__name__}")
print(f"后端 name: {backend.name}")


工厂返回: PlaceholderBackend
后端 name: placeholder


## 4. 带 fallback 的配置合并

Clawith 有一个很实用的设计：`from_dict()` 方法支持双层配置合并——  
工具级配置优先，缺失的字段从环境变量配置回退。


In [5]:
def merge_sandbox_config(
    tool_config: dict | None,
    fallback_config: SandboxConfig,
) -> SandboxConfig:
    """合并工具配置和回退配置（工具配置优先）。"""
    if tool_config is None:
        return fallback_config

    kwargs = {
        "sandbox_type": SandboxType(
            tool_config.get("sandbox_type", fallback_config.sandbox_type.value)
        ),
        "cpu_limit": tool_config.get("cpu_limit", fallback_config.cpu_limit),
        "memory_limit": tool_config.get("memory_limit", fallback_config.memory_limit),
        "allow_network": tool_config.get("allow_network", fallback_config.allow_network),
        "default_timeout": tool_config.get("default_timeout", fallback_config.default_timeout),
        "max_timeout": tool_config.get("max_timeout", fallback_config.max_timeout),
    }
    return SandboxConfig(**kwargs)


# 环境变量级的默认配置（相当于 .env 中的 SANDBOX_*）
env_config = SandboxConfig(allow_network=False, cpu_limit="0.5")
print(f"环境配置: {env_config}")

# 某个 Agent 的工具级配置（覆盖了 allow_network）
tool_cfg_dict = {"allow_network": True, "cpu_limit": "1.0"}
merged = merge_sandbox_config(tool_cfg_dict, env_config)
print(f"合并后 : {merged}")
# allow_network 被工具配置覆盖为 True，其余保持环境配置
assert merged.allow_network == True
assert merged.cpu_limit == "1.0"
assert merged.memory_limit == "256m"  # 未覆盖，保持环境值
print("✅ 字段级 fallback 工作正常")


环境配置: SandboxConfig(type=subprocess, network=off, cpu=0.5, mem=256m)
合并后 : SandboxConfig(type=subprocess, network=on, cpu=1.0, mem=256m)
✅ 字段级 fallback 工作正常


## 5. 对比 Clawith 真实实现

打开源码对照：

| 本 Notebook | Clawith 文件 | 行号 |
|---|---|---|
| `SandboxBackend` Protocol | `base.py` | 31-79 |
| `ExecutionResult` | `base.py` | 8-17 |
| `SandboxType` | `config.py` | 9-18 |
| `SandboxConfig` | `config.py` | 21-49 |
| `_BACKEND_REGISTRY` | `registry.py` | 38 |
| `get_sandbox_backend()` | `registry.py` | 10-33 |
| `merge_sandbox_config` → `from_dict()` | `config.py` | 52-109 |

你应该注意到：

- Clawith 的 `SandboxConfig` 继承自 Pydantic `BaseModel`（有自动校验和序列化）
- `from_dict()` 还支持**解密敏感字段**（如 `api_key`）
- `BaseSandboxBackend` ABC 提供了 `_format_result()` 公共方法
- 注册表在模块加载时自动注册内置后端（`registry.py:64-84`）

---

**小结：** 接口、配置、注册表——这三样东西构成了沙盒系统"可插拔"的骨架。  
下一份 Notebook 会填充真正的执行逻辑。
